# 09d Final Daily-Origin LightGBM Evaluation

This notebook runs the final daily-origin rolling-origin evaluation for the thesis feature sets M1, M2 and M3. M1 is the no-weather baseline, M2 is the operational forecast-weather model, and M3 is the realised-weather perfect-information benchmark. M4 is excluded from the final thesis comparison.

The evaluation uses daily forecast origins over the reserved test period. For each origin and horizon, the training window expands over rows with target dates before the forecast origin. The M2 medium-range weather input uses the conservative `stronger_climatology_drift` scenario: h=0..2 use archived deterministic MEPS forecasts, while h=3..10 use the synthetic emulator drifting toward same-calendar-day local climatology with `alpha(10) = 0.25`.

The heavy rolling-origin loop is controlled by `EXECUTE_FINAL_RUN`. Setup, validation and preflight cells document the run configuration before model fitting is launched.


## Configuration

This cell defines the final-run switches, daily-origin evaluation window, feature sets, horizons, conservative M2 weather scenario and output identity. `FEATURE_SETS` remains the canonical M1/M2/M3 comparison set; `FEATURE_SETS_TO_RUN` controls which shards are built in the current resumable execution.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

# Final-run switches.
# EXECUTE_FINAL_RUN controls the heavy rolling-origin training loop.
# False allows setup, validation and preflight without model fitting.
EXECUTE_FINAL_RUN = True
# Existing final outputs are overwritten only when FORCE_OVERWRITE is true.
FORCE_OVERWRITE = True

RUN_MODE = "full"
ORIGIN_SCHEDULE_MODE = "daily"
EVAL_START_DATE = "2024-08-01"
EVAL_END_DATE = "2025-07-31"
ORIGIN_STEP_DAYS = 5  # unused for daily schedule; kept for compatibility/printing

FEATURE_SETS = ["M1", "M2", "M3"]  # M4 intentionally excluded (canonical full set - do NOT trim)
# Feature-set shard switch for resumable execution.
# The training loop fits exactly the sets listed here.
# A subset can be used for a partial rerun; completed shards are reused later.
# The list must remain a non-empty subset of FEATURE_SETS.
# Restore to ["M1", "M2", "M3"] before producing the final combined delivery run.
FEATURE_SETS_TO_RUN = ["M2"]
HORIZONS_TO_RUN = list(range(0, 11))
MAX_ORIGINS = None
MODEL_FAMILY = "lightgbm_tuned"
RANDOM_STATE = 42
N_JOBS = -1
EST_MINUTES_PER_FIT = 0.42
CLIP_NEGATIVE_PREDICTIONS = True
SAVE_ALL_FITTED_MODELS_FOR_SHAP = False

# Conservative M2 medium-range weather emulator.
# Reuse the tested stronger_climatology_drift schedule from notebook 09b.
# h<=2 uses deterministic MEPS; h=3..10 follows the conservative alpha schedule.
M2_WEATHER_SCENARIO = "stronger_climatology_drift"
CONSERVATIVE_ALPHA_H10 = 0.25  # exact target at h=10
CONSERVATIVE_ALPHA_H3_EXPECTED = CONSERVATIVE_ALPHA_H10 ** (1.0 / 8.0)

# Output identity.
RUN_TAG = "final_daily_m1_m2_m3"
OUTPUT_SUBDIR_NAME = "final_daily_m1_m2_m3"
SCHEDULE_TAG = RUN_TAG
FILE_SUFFIX = f"_{RUN_TAG}"
FEATURE_SET_SCOPE = "_".join(FEATURE_SETS).lower()
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_CREATED_AT_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = f"lightgbm_rolling_origin__{SCHEDULE_TAG}__{FEATURE_SET_SCOPE}__{RUN_TIMESTAMP_UTC}"

assert RUN_MODE == "full"
assert ORIGIN_SCHEDULE_MODE == "daily", "09d is the daily-origin final run."
assert FEATURE_SETS == ["M1", "M2", "M3"], "09d evaluates exactly M1, M2, M3."
assert "M4" not in FEATURE_SETS, "M4 is excluded from the final thesis evaluation."
assert FEATURE_SETS_TO_RUN and set(FEATURE_SETS_TO_RUN).issubset(FEATURE_SETS), (
    "FEATURE_SETS_TO_RUN must be a non-empty subset of FEATURE_SETS."
)
print("Config OK:", RUN_ID)
print(
    "FEATURE_SETS:", FEATURE_SETS,
    "| TO_RUN:", FEATURE_SETS_TO_RUN,
    "| schedule:", ORIGIN_SCHEDULE_MODE,
    "| EXECUTE_FINAL_RUN:", EXECUTE_FINAL_RUN,
)

## Environment, paths and input checks

This cell resolves project-relative paths, requires the upstream ML panel, feature registry, tuned LightGBM parameters and conservative M2 weather variant, and creates the isolated final daily-origin output folders.


In [ ]:
import gc
import json
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from inside the project folder.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_TUNING_DIR = OUTPUT_DIR / "lightgbm_tuning"
OUTPUT_ROLLING_DIR = OUTPUT_DIR / "lightgbm_rolling_origin"
OUTPUT_ROLLING_RUN_DIR = OUTPUT_ROLLING_DIR / OUTPUT_SUBDIR_NAME  # final_daily_m1_m2_m3
FIG_DIR = OUTPUT_ROLLING_RUN_DIR / f"figures{FILE_SUFFIX}"
PREFLIGHT_AUDIT_DIR = OUTPUT_ROLLING_RUN_DIR / "preflight_audit"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"
BEST_PARAMS_PATH = OUTPUT_TUNING_DIR / "lightgbm_optuna_best_params_full.json"

# Accepted baseline operational weather file; read-only in this notebook.
BASELINE_OPERATIONAL_PATH = PROCESSED_DIR / "weather_forecast_operational_windows.parquet"
# Tested stronger_climatology_drift variant from notebook 09b.
SRC_CONSERVATIVE_VARIANT = (
    OUTPUT_DIR
    / "weather_emulator_alpha_sensitivity"
    / "m2_alpha_sensitivity"
    / "weather_forecast_operational_windows__stronger_climatology_drift.parquet"
)
# Final 09d weather artifact, created from the 09b variant if missing.
FINAL_CONSERVATIVE_WEATHER_PATH = OUTPUT_ROLLING_RUN_DIR / "final_weather_forecast_operational_windows_conservative.parquet"

OUTPUT_ROLLING_RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
# Crash-resumable checkpoint folders, one shard per feature set and horizon.
PREDICTION_SHARDS_DIR = OUTPUT_ROLLING_RUN_DIR / "prediction_shards"
METRICS_SHARDS_DIR = OUTPUT_ROLLING_RUN_DIR / "metrics_shards"
PREDICTION_SHARDS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_SHARDS_DIR.mkdir(parents=True, exist_ok=True)


def shard_pred_path(label, h):
    return PREDICTION_SHARDS_DIR / f"predictions_{label}_h{int(h):02d}.parquet"

def shard_metrics_path(label, h):
    return METRICS_SHARDS_DIR / f"metrics_{label}_h{int(h):02d}.csv"

def expected_shard_keys():
    return [(label, h) for label in FEATURE_SETS for h in HORIZONS_TO_RUN]


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def require_file(path, desc):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {desc}: {project_relative(path)}.")


for path, desc in [
    (ML_PANEL_PATH, "ML forecast panel (notebook 06)"),
    (FEATURE_REGISTRY_PATH, "feature registry (notebook 06)"),
    (BEST_PARAMS_PATH, "tuned LightGBM parameters (notebook 08, full)"),
    (SRC_CONSERVATIVE_VARIANT, "stronger_climatology_drift weather variant (notebook 09b)"),
]:
    require_file(path, desc)


def out_path(stem, ext):
    return OUTPUT_ROLLING_RUN_DIR / f"lightgbm_rolling_origin_{stem}{FILE_SUFFIX}.{ext}"


PREDICTIONS_PATH = out_path("predictions", "parquet")
METRICS_PATH = out_path("metrics", "csv")
METRICS_BY_HFS_PATH = out_path("metrics_by_feature_horizon", "csv")
GAIN_SUMMARY_PATH = out_path("gain_summary", "csv")
RUNTIME_SUMMARY_PATH = out_path("runtime_summary", "csv")
VALIDATION_CHECKS_PATH = out_path("validation_checks", "csv")
RUN_CONFIG_PATH = out_path("run_config", "json")
USED_PARAMS_JSON_PATH = out_path("used_params", "json")
FEATURE_REGISTRY_SNAPSHOT_PATH = out_path("feature_registry_snapshot", "csv")
ALPHA_SCHEDULE_PATH = out_path("alpha_schedule", "csv")

# Final outputs must stay inside the final daily-origin folder.
FORBIDDEN_OUTPUT_DIRS = [
    OUTPUT_ROLLING_DIR,
    OUTPUT_ROLLING_DIR / "m2_daily_origin",
    OUTPUT_DIR / "weather_emulator_alpha_sensitivity",
]
for p in [PREDICTIONS_PATH, METRICS_PATH, RUN_CONFIG_PATH, FINAL_CONSERVATIVE_WEATHER_PATH]:
    assert p.parent == OUTPUT_ROLLING_RUN_DIR, f"Output {p} must be under {OUTPUT_ROLLING_RUN_DIR}"
    assert "full_step5" not in p.name, "Final outputs must not reuse the full_step5 tag."

print("Project root:", PROJECT_ROOT)
print("Final output folder:", project_relative(OUTPUT_ROLLING_RUN_DIR))
print("ML panel:", project_relative(ML_PANEL_PATH))
print("Conservative weather source (09b):", project_relative(SRC_CONSERVATIVE_VARIANT))
print("Final conservative weather artifact:", project_relative(FINAL_CONSERVATIVE_WEATHER_PATH))
print("Baseline operational (read-only, never written):", project_relative(BASELINE_OPERATIONAL_PATH))

## Conservative M2 alpha schedule and weather artifact

This cell records the conservative M2 alpha schedule and materializes the final daily-origin weather artifact from the tested 09b `stronger_climatology_drift` variant. The accepted baseline operational weather file is not modified.


In [ ]:
def conservative_alpha(h, alpha_h10=CONSERVATIVE_ALPHA_H10):
    """alpha(h)=1.0 for h<=2 (actual MEPS); exponential drift to climatology for h=3..10."""
    h = int(h)
    if h <= 2:
        return 1.0
    return float(alpha_h10 ** ((h - 2) / 8.0))


alpha_rows = []
for h in HORIZONS_TO_RUN:
    a = conservative_alpha(h)
    alpha_rows.append(
        {
            "horizon": h,
            "alpha": a,
            "climatology_weight": 1.0 - a,
            "horizon_tier": "actual_meps_h0_h2" if h <= 2 else "synthetic_scenario_h3_h10",
            "weather_source": (
                "deterministic_meps" if h <= 2 else "synthetic_conservative_climatology_drift"
            ),
        }
    )
alpha_schedule = pd.DataFrame(alpha_rows)
alpha_schedule.to_csv(ALPHA_SCHEDULE_PATH, index=False)
print("Conservative M2 alpha schedule (scenario:", M2_WEATHER_SCENARIO, "):")
print(alpha_schedule.to_string(index=False))
print("Saved alpha schedule:", project_relative(ALPHA_SCHEDULE_PATH))

# Materialize the final conservative weather artifact from the 09b variant.
assert SRC_CONSERVATIVE_VARIANT != BASELINE_OPERATIONAL_PATH
assert FINAL_CONSERVATIVE_WEATHER_PATH != BASELINE_OPERATIONAL_PATH
if not FINAL_CONSERVATIVE_WEATHER_PATH.exists():
    shutil.copy2(SRC_CONSERVATIVE_VARIANT, FINAL_CONSERVATIVE_WEATHER_PATH)
    print(
        "Created final conservative weather artifact from 09b variant:",
        project_relative(FINAL_CONSERVATIVE_WEATHER_PATH),
    )
else:
    print(
        "Final conservative weather artifact already present:",
        project_relative(FINAL_CONSERVATIVE_WEATHER_PATH),
    )
CONSERVATIVE_WEATHER_PATH = FINAL_CONSERVATIVE_WEATHER_PATH

## Feature registry and feature sets

This cell loads the feature registry, constructs the registry-driven M1, M2 and M3 feature sets, and writes a feature-registry snapshot for reproducibility.


In [ ]:
feature_registry = pd.read_csv(FEATURE_REGISTRY_PATH)


def _to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


for col in [
    "allowed_m1_baseline",
    "allowed_m2_synthetic_point_weather",
    "allowed_m3_perfect_information",
    "allowed_m4_synthetic_engineered_weather",
    "leakage_risk",
    "known_at_origin",
]:
    feature_registry[col] = _to_bool(feature_registry[col])

KEY_COLUMNS = feature_registry.loc[feature_registry["role"] == "key", "column"].tolist()
TARGET_COLUMNS = feature_registry.loc[
    feature_registry["role"].isin(["target", "robustness_target"]), "column"
].tolist()
CATEGORICAL_FEATURE_CANDIDATES = [
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "HelligdagNavn",
    "season",
]
FORBIDDEN_FEATURE_COLUMNS = set(KEY_COLUMNS + TARGET_COLUMNS + ["origin_season"]) - {"AvdelingID", "Analyse_Kategori"}


def feature_set_columns(label):
    flag = {
        "M1": "allowed_m1_baseline",
        "M2": "allowed_m2_synthetic_point_weather",
        "M3": "allowed_m3_perfect_information",
        "M4": "allowed_m4_synthetic_engineered_weather",
    }[label]
    reg = feature_registry.loc[feature_registry[flag], "column"].tolist()
    reg = [c for c in reg if c not in FORBIDDEN_FEATURE_COLUMNS]
    explicit = [c for c in CATEGORICAL_FEATURE_CANDIDATES if c in {"AvdelingID", "Analyse_Kategori"}]
    return list(dict.fromkeys(explicit + reg))


feature_sets = {label: feature_set_columns(label) for label in FEATURE_SETS}
feature_registry.to_csv(FEATURE_REGISTRY_SNAPSHOT_PATH, index=False)


def split_numeric_categorical(columns):
    cat = [c for c in columns if c in CATEGORICAL_FEATURE_CANDIDATES]
    num = [c for c in columns if c not in CATEGORICAL_FEATURE_CANDIDATES]
    return num, cat


feature_set_numeric, feature_set_categorical = {}, {}
for label, cols in feature_sets.items():
    n, c = split_numeric_categorical(cols)
    feature_set_numeric[label] = n
    feature_set_categorical[label] = c
    print(f"{label}: {len(cols)} features ({len(n)} numeric, {len(c)} categorical)")

## Tuned LightGBM hyperparameters

This cell loads the validation-selected LightGBM parameters from notebook 08, removes helper-only tuning keys, and writes the exact parameter payload used in this final evaluation.


In [ ]:
with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    tuning = json.load(f)
static_kwargs = dict(tuning.get("lightgbm_static_kwargs", {}))
static_kwargs.setdefault("objective", "regression")
static_kwargs.setdefault("n_jobs", N_JOBS)
static_kwargs.setdefault("random_state", RANDOM_STATE)
static_kwargs.setdefault("verbose", -1)
feature_set_params = {label: dict(tuning["feature_sets"][label]["best_params"]) for label in FEATURE_SETS}
for d in feature_set_params.values():
    d.pop("use_max_depth", None)
    d.setdefault("max_depth", -1)
USED_PARAMS_JSON_PATH.write_text(
    json.dumps(
        {"static_kwargs": static_kwargs, "feature_set_params": feature_set_params},
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)
print("Loaded tuned params for:", list(feature_set_params))
print("Saved used params:", project_relative(USED_PARAMS_JSON_PATH))

## Load ML panel and inject conservative M2 weather

The ML panel is loaded with the required keys, target, feature columns and optional audit columns. Forecast-weather columns used by M2 are overlaid from the conservative `stronger_climatology_drift` weather artifact by `(AvdelingID, origin_date, target_date, horizon)`. M1 and M3 feature columns are checked to ensure that the overlay does not affect them.


In [ ]:
available_columns = set(pq.ParquetFile(ML_PANEL_PATH).schema_arrow.names)
OPTIONAL_AUDIT_COLUMNS = [
    "Closed",
    "is_open",
    "Ukedag",
    "Helligdager",
    "HelligdagNavn",
    "target_year",
    "target_month",
    "target_day",
    "target_iso_week",
    "origin_datetime_utc",
    "origin_hour",
]
OPTIONAL_AUDIT_COLUMNS_PRESENT = [c for c in OPTIONAL_AUDIT_COLUMNS if c in available_columns]
selected_columns = sorted(
    set(
        KEY_COLUMNS
        + ["Antall_total"]
        + [c for label in FEATURE_SETS for c in feature_sets[label]]
        + OPTIONAL_AUDIT_COLUMNS_PRESENT
    )
)
missing_required = sorted(set(KEY_COLUMNS + ["Antall_total"]) - available_columns)
if missing_required:
    raise AssertionError(f"ML panel missing required columns: {missing_required}")

panel = pd.read_parquet(ML_PANEL_PATH, columns=selected_columns)
panel["target_date"] = pd.to_datetime(panel["target_date"]).dt.normalize()
panel["origin_date"] = pd.to_datetime(panel["origin_date"]).dt.normalize()
panel["horizon"] = panel["horizon"].astype("int8")
panel = panel.loc[panel["horizon"].isin(HORIZONS_TO_RUN)].reset_index(drop=True)
panel = panel.loc[panel["Antall_total"].notna()].reset_index(drop=True)
if "origin_season" in panel.columns:
    panel = panel.drop(columns=["origin_season"])

# Conservative M2 weather overlay, aligned with notebook 09b.
POINT_COLUMNS = {
    "temp": "temp_fcst_mean",
    "wind": "wind_fcst_mean",
    "humid": "humid_fcst_mean",
    "cloud": "cloud_fcst_mean",
    "precip": "precip_fcst_sum",
}
weather_point_cols = [POINT_COLUMNS[v] for v in POINT_COLUMNS]
weather_uncert_cols = [
    c for c in panel.columns
    if c.endswith(("_fcst_std", "_fcst_p10", "_fcst_p90", "_fcst_interval_width"))
]
weather_aux_cols = [
    c for c in [
        "precip_amount_uncertainty",
        "precip_wet_amount_p90",
        "precip_wet_prob",
        "cloud_open_prob",
        "cloud_partly_prob",
        "cloud_overcast_prob",
    ]
    if c in panel.columns
]
weather_replace_cols = [
    c for c in dict.fromkeys(weather_point_cols + weather_uncert_cols + weather_aux_cols)
    if c in panel.columns
]

# Replaced forecast-weather columns must belong only to M2.
assert not (set(weather_replace_cols) & set(feature_sets["M1"])), "weather overlay touches M1 features"
m3_overlap = set(weather_replace_cols) & set(feature_sets["M3"])
assert not m3_overlap, f"weather overlay touches M3 features: {sorted(m3_overlap)}"

variant_available = set(pq.ParquetFile(CONSERVATIVE_WEATHER_PATH).schema_arrow.names)
variant = pd.read_parquet(
    CONSERVATIVE_WEATHER_PATH,
    columns=[
        "AvdelingID",
        "aggregation_window",
        "target_date",
        "horizon_days",
        "origin_date",
    ] + [c for c in weather_replace_cols if c in variant_available],
)
variant["target_date"] = pd.to_datetime(variant["target_date"]).dt.normalize()
variant["origin_date"] = pd.to_datetime(variant["origin_date"]).dt.normalize()
variant = variant.loc[variant["aggregation_window"] == "trade_08_19"].copy()
variant = variant.rename(columns={"horizon_days": "horizon"})
variant["horizon"] = variant["horizon"].astype("int8")
variant = variant.loc[variant["horizon"].isin(HORIZONS_TO_RUN)].copy()

keep_keys = ["AvdelingID", "origin_date", "target_date", "horizon"]
overlay_cols = [c for c in weather_replace_cols if c in variant.columns]
rows_before = len(panel)
panel = panel.drop(columns=overlay_cols).merge(
    variant[keep_keys + overlay_cols],
    on=keep_keys,
    how="left",
)
assert len(panel) == rows_before, "conservative weather overlay changed row count"
CONSERVATIVE_OVERLAY_COLS = overlay_cols
print(
    f"Conservative M2 weather overlaid: {len(overlay_cols)} columns from "
    f"{project_relative(CONSERVATIVE_WEATHER_PATH)}"
)
print(f"Panel rows after filters: {len(panel):,}")

eval_start_ts = pd.Timestamp(EVAL_START_DATE).normalize()
eval_end_ts = pd.Timestamp(EVAL_END_DATE).normalize()


def build_origin_schedule(panel_, eval_start, eval_end, mode, step_days, max_origins=None):
    available = sorted(
        pd.to_datetime(
            panel_.loc[
                (panel_["origin_date"] >= eval_start) & (panel_["origin_date"] <= eval_end),
                "origin_date",
            ]
            .dropna()
            .unique()
        ).normalize()
    )
    available_set = set(available)
    if mode == "daily":
        schedule = [d for d in pd.date_range(eval_start, eval_end, freq="D") if d in available_set]
    elif mode == "step":
        schedule = []
        cursor = eval_start
        while cursor <= eval_end:
            if cursor in available_set:
                schedule.append(cursor)
            cursor = cursor + pd.Timedelta(days=int(step_days))
    else:
        raise ValueError(f"Unsupported ORIGIN_SCHEDULE_MODE: {mode!r}")
    return schedule[:int(max_origins)] if max_origins is not None else schedule


origin_schedule = build_origin_schedule(
    panel,
    eval_start_ts,
    eval_end_ts,
    ORIGIN_SCHEDULE_MODE,
    ORIGIN_STEP_DAYS,
    MAX_ORIGINS,
)
panel = panel.sort_values(["target_date", "horizon"]).reset_index(drop=True)
print(
    f"Daily origin schedule length: {len(origin_schedule)} "
    f"({origin_schedule[0].date()} .. {origin_schedule[-1].date()})"
)

## Final-run validation

This cell verifies the final evaluation configuration before the heavy loop: M1/M2/M3 only, M4 excluded, daily-origin scheduling, isolated output paths, conservative alpha values, unchanged baseline weather input, and shard checkpoint status.


In [ ]:
checks = []


def _chk(name, passed, detail=""):
    checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})
    print(("OK  " if passed else "FAIL") + f"  {name}  {detail}")

a3 = conservative_alpha(3)
a10 = conservative_alpha(10)
_chk("feature_sets_exactly_m1_m2_m3", FEATURE_SETS == ["M1", "M2", "M3"], FEATURE_SETS)
_chk("m4_excluded", "M4" not in FEATURE_SETS)
_chk("origin_schedule_daily", ORIGIN_SCHEDULE_MODE == "daily")
_chk(
    "outputs_under_final_folder",
    all(p.parent == OUTPUT_ROLLING_RUN_DIR for p in [PREDICTIONS_PATH, METRICS_PATH, RUN_CONFIG_PATH])
    and OUTPUT_SUBDIR_NAME == "final_daily_m1_m2_m3",
    project_relative(OUTPUT_ROLLING_RUN_DIR),
)
_existing = [p for p in [PREDICTIONS_PATH, METRICS_PATH] if p.exists()]
_chk(
    "no_overwrite_without_force",
    (not _existing) or FORCE_OVERWRITE,
    f"existing={[project_relative(p) for p in _existing]}; FORCE_OVERWRITE={FORCE_OVERWRITE}",
)
_chk("alpha_schedule_saved", ALPHA_SCHEDULE_PATH.exists(), project_relative(ALPHA_SCHEDULE_PATH))
_chk("alpha_h10_equals_0.25", abs(a10 - 0.25) < 1e-9, f"alpha_h10={a10:.6f}")
_chk(
    "alpha_h3_matches_conservative_schedule",
    abs(a3 - CONSERVATIVE_ALPHA_H3_EXPECTED) < 1e-9,
    f"alpha_h3={a3:.4f} ~ 0.8409 (expected {CONSERVATIVE_ALPHA_H3_EXPECTED:.4f})",
)
_chk(
    "h0_h2_use_actual_meps",
    all(conservative_alpha(h) == 1.0 for h in (0, 1, 2)),
    "alpha=1.0 at h=0,1,2 -> deterministic MEPS features unchanged",
)
_chk(
    "h3_h10_use_conservative_synthetic",
    all(conservative_alpha(h) < 1.0 for h in range(3, 11)) and len(CONSERVATIVE_OVERLAY_COLS) > 0,
    f"overlaid {len(CONSERVATIVE_OVERLAY_COLS)} conservative weather cols for h=3..10",
)
_chk(
    "baseline_operational_not_targeted",
    FINAL_CONSERVATIVE_WEATHER_PATH != BASELINE_OPERATIONAL_PATH
    and CONSERVATIVE_WEATHER_PATH != BASELINE_OPERATIONAL_PATH,
)

# Shard checkpoint preflight.
_exp_shards = expected_shard_keys()
_completed = [
    (l, h) for (l, h) in _exp_shards
    if shard_pred_path(l, h).exists() and shard_metrics_path(l, h).exists()
]
_missing = [(l, h) for (l, h) in _exp_shards if (l, h) not in _completed]
_chk(
    "expected_shard_count_is_33",
    len(_exp_shards) == len(FEATURE_SETS) * len(HORIZONS_TO_RUN) == 33,
    f"{len(_exp_shards)} shards = {len(FEATURE_SETS)} feature sets x {len(HORIZONS_TO_RUN)} horizons",
)
_chk(
    "final_outputs_not_overwritten_without_force",
    (not (PREDICTIONS_PATH.exists() or METRICS_PATH.exists())) or FORCE_OVERWRITE,
    f"FORCE_OVERWRITE={FORCE_OVERWRITE}",
)
print(
    f"Checkpoint shards: completed {len(_completed)}/{len(_exp_shards)}; missing {len(_missing)} "
    f"(FORCE_OVERWRITE={FORCE_OVERWRITE} -> "
    f"{'all will be rebuilt' if FORCE_OVERWRITE else 'completed shards will be skipped'})"
)
print("Prediction shard dir:", project_relative(PREDICTION_SHARDS_DIR))
print("Metrics shard dir:", project_relative(METRICS_SHARDS_DIR))
print("Example expected shard paths:")
for (l, h) in _exp_shards[:3] + _exp_shards[-2:]:
    print("  ", project_relative(shard_pred_path(l, h)), "|", project_relative(shard_metrics_path(l, h)))

validation_df = pd.DataFrame(checks)
validation_df.to_csv(VALIDATION_CHECKS_PATH, index=False)
print("\nAll validation checks passed:", bool(validation_df["passed"].all()))
assert validation_df["passed"].all(), "09d validation failed; resolve before running."

## Pipeline, metrics, preflight audit and run config

This cell defines the LightGBM pipeline and forecast-error metrics, writes preflight diagnostics, estimates the number of fits and runtime, and saves the run configuration used for reproducibility.


In [ ]:
def build_pipeline(num_cols, cat_cols, params):
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    transformer = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_cols),
            ("cat", encoder, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    from lightgbm import LGBMRegressor

    full = {**static_kwargs, **params}
    return Pipeline([("preprocess", transformer), ("model", LGBMRegressor(**full))])


def rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt, "float64") - np.asarray(yp, "float64")) ** 2)))


def mae(yt, yp):
    return float(np.mean(np.abs(np.asarray(yt, "float64") - np.asarray(yp, "float64"))))


def wape(yt, yp):
    d = float(np.sum(np.abs(np.asarray(yt, "float64"))))
    if d > 0:
        return float(np.sum(np.abs(np.asarray(yt, "float64") - np.asarray(yp, "float64"))) / d)
    return float("nan")


def bias(yt, yp):
    return float(np.mean(np.asarray(yp, "float64") - np.asarray(yt, "float64")))


# Preflight audit and runtime estimate.
block = panel.loc[
    panel["origin_date"].isin(origin_schedule)
    & panel["horizon"].isin(HORIZONS_TO_RUN)
    & (panel["target_date"] <= eval_end_ts)
].copy()
PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
if not block.empty:
    tgt = (
        block.groupby("horizon", observed=True)["Antall_total"]
        .agg(n_rows="count", mean="mean", zero_share=lambda s: float((s <= 0).mean()))
        .reset_index()
    )
    tgt.to_csv(PREFLIGHT_AUDIT_DIR / f"target_summary_by_horizon_{RUN_MODE}.csv", index=False)
n_fits = int(len(origin_schedule) * len(HORIZONS_TO_RUN) * len(FEATURE_SETS))
est_hours = n_fits * EST_MINUTES_PER_FIT / 60.0
pd.DataFrame([
    {
        "run_id": RUN_ID,
        "schedule_tag": SCHEDULE_TAG,
        "n_origins": len(origin_schedule),
        "n_horizons": len(HORIZONS_TO_RUN),
        "n_feature_sets": len(FEATURE_SETS),
        "estimated_n_fits": n_fits,
        "est_minutes_per_fit": EST_MINUTES_PER_FIT,
        "estimated_runtime_hours": est_hours,
    }
]).to_csv(PREFLIGHT_AUDIT_DIR / f"estimated_runtime_{RUN_MODE}.csv", index=False)
print(
    f"Estimated fits: {n_fits:,}  "
    f"(origins {len(origin_schedule)} x horizons {len(HORIZONS_TO_RUN)} x feature sets {len(FEATURE_SETS)})"
)
print(
    f"Estimated runtime: {est_hours:.1f} hours at {EST_MINUTES_PER_FIT:.2f} min/fit "
    "(approximate; daily fits grow with the expanding window)"
)
print(f"Preflight audit saved under {project_relative(PREFLIGHT_AUDIT_DIR)}")

run_config_payload = {
    "run_id": RUN_ID,
    "schedule_tag": SCHEDULE_TAG,
    "run_created_at_utc": RUN_CREATED_AT_UTC,
    "run_mode": RUN_MODE,
    "eval_start_date": EVAL_START_DATE,
    "eval_end_date": EVAL_END_DATE,
    "origin_schedule_mode": ORIGIN_SCHEDULE_MODE,
    "feature_sets": FEATURE_SETS,
    "horizons": HORIZONS_TO_RUN,
    "model_family": MODEL_FAMILY,
    "m2_weather_scenario": M2_WEATHER_SCENARIO,
    "conservative_alpha_h10": CONSERVATIVE_ALPHA_H10,
    "conservative_alpha_h3": conservative_alpha(3),
    "conservative_weather_artifact": project_relative(CONSERVATIVE_WEATHER_PATH),
    "conservative_weather_source": project_relative(SRC_CONSERVATIVE_VARIANT),
    "baseline_operational_left_unchanged": project_relative(BASELINE_OPERATIONAL_PATH),
    "clip_negative_predictions": bool(CLIP_NEGATIVE_PREDICTIONS),
    "save_all_fitted_models_for_shap": bool(SAVE_ALL_FITTED_MODELS_FOR_SHAP),
    "estimated_n_fits": n_fits,
    "estimated_runtime_hours": est_hours,
    "execute_final_run_flag_default": False,
    "output_paths": {
        "predictions": project_relative(PREDICTIONS_PATH),
        "metrics": project_relative(METRICS_PATH),
        "metrics_by_feature_horizon": project_relative(METRICS_BY_HFS_PATH),
        "gain_summary": project_relative(GAIN_SUMMARY_PATH),
        "run_config": project_relative(RUN_CONFIG_PATH),
        "alpha_schedule": project_relative(ALPHA_SCHEDULE_PATH),
        "prediction_shards_dir": project_relative(PREDICTION_SHARDS_DIR),
        "metrics_shards_dir": project_relative(METRICS_SHARDS_DIR),
    },
}
RUN_CONFIG_PATH.write_text(json.dumps(run_config_payload, indent=2, default=str), encoding="utf-8")
print("Saved run config:", project_relative(RUN_CONFIG_PATH))

## Rolling-origin training loop

This guarded cell performs the heavy daily-origin evaluation when `EXECUTE_FINAL_RUN` is true. It fits one model per feature set, horizon and origin, writes crash-resumable shards, and combines complete shards into the final prediction, metric, gain-summary and runtime outputs.


In [ ]:
def run_final_evaluation():
    """Crash-resumable rolling-origin evaluation, checkpointed by feature_set x horizon.

    Outer loop: feature_set; second loop: horizon; inner loop: origin. One prediction parquet and one
    metrics CSV are written per (feature_set, horizon) shard. A completed shard (both files present) is
    skipped unless FORCE_OVERWRITE. Only one shard is held in memory at a time. After all 33 shards exist
    they are combined into the final predictions/metrics plus the pooled and gain summaries.
    """
    KEEP_PRED = ["AvdelingID", "Analyse_Kategori", "origin_date", "target_date", "horizon"]
    expected = expected_shard_keys()
    t0 = time.time()
    built = []
    skipped = []
    n_shards_total = len(FEATURE_SETS_TO_RUN) * len(HORIZONS_TO_RUN)  # X of N denominator for this run
    shard_no = 0
    PROGRESS_EVERY = 25  # throttle: one cheap flushed heartbeat per 25 origin-fits

    for label in FEATURE_SETS_TO_RUN:
        num_cols, cat_cols = feature_set_numeric[label], feature_set_categorical[label]
        params = feature_set_params[label]
        for h in HORIZONS_TO_RUN:
            ppath, mpath = shard_pred_path(label, h), shard_metrics_path(label, h)
            shard_no += 1
            if ppath.exists() and mpath.exists() and not FORCE_OVERWRITE:
                skipped.append((label, h))
                print(
                    f"  SKIP  [{shard_no:02d}/{n_shards_total}] "
                    f"{label} h{int(h):02d} (shard already complete)",
                    flush=True,
                )
                continue
            shard_pred = []
            shard_metrics = []
            n_origins = len(origin_schedule)
            print(
                f"  START [{shard_no:02d}/{n_shards_total}] {label} h{int(h):02d}: "
                f"up to {n_origins} origins",
                flush=True,
            )
            for oi, origin in enumerate(origin_schedule, start=1):
                train_h = panel.loc[
                    (panel["target_date"] < origin) & (panel["horizon"] == h)
                ]
                test_h = panel.loc[
                    (panel["origin_date"] == origin)
                    & (panel["target_date"] <= eval_end_ts)
                    & (panel["horizon"] == h)
                ]
                if train_h.empty or test_h.empty:
                    continue
                pipeline = build_pipeline(num_cols, cat_cols, params)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pipeline.fit(
                        train_h[num_cols + cat_cols],
                        train_h["Antall_total"].astype("float32").to_numpy(),
                    )
                y_pred = np.asarray(pipeline.predict(test_h[num_cols + cat_cols]), dtype="float64")
                if CLIP_NEGATIVE_PREDICTIONS:
                    y_pred = np.clip(y_pred, 0.0, None)
                y_true = test_h["Antall_total"].astype("float64").to_numpy()
                pf = test_h[KEEP_PRED].copy()
                pf["run_mode"] = RUN_MODE
                pf["model_family"] = MODEL_FAMILY
                pf["feature_set"] = label
                pf["y_true"] = y_true
                pf["y_pred"] = y_pred
                pf["absolute_error"] = np.abs(y_true - y_pred)
                pf["squared_error"] = (y_true - y_pred) ** 2
                pf["horizon_tier"] = np.where(pf["horizon"] <= 2, "actual_meps_h0_h2", "synthetic_scenario_h3_h10")
                shard_pred.append(pf)
                shard_metrics.append(
                    {
                        "feature_set": label,
                        "origin_date": origin.date().isoformat(),
                        "horizon": int(h),
                        "n_train": int(len(train_h)),
                        "n_test": int(len(test_h)),
                        "rmse": rmse(y_true, y_pred),
                        "mae": mae(y_true, y_pred),
                        "wape": wape(y_true, y_pred),
                        "bias": bias(y_true, y_pred),
                    }
                )
                del pipeline
                if oi % PROGRESS_EVERY == 0 or oi == n_origins:
                    print(
                        f"    [{shard_no:02d}/{n_shards_total}] {label} h{int(h):02d}: "
                        f"origin {oi}/{n_origins} | fits {len(shard_pred)} "
                        f"| {(time.time() - t0) / 60:.1f} min",
                        flush=True,
                    )
            if not shard_pred:
                print(
                    f"  WARN  [{shard_no:02d}/{n_shards_total}] {label} h{int(h):02d}: "
                    "no eval rows; shard not written",
                    flush=True,
                )
                continue
            # Write temporary shard files before atomic replacement.
            tmp_p = ppath.with_suffix(ppath.suffix + ".tmp")
            tmp_m = mpath.with_suffix(mpath.suffix + ".tmp")
            sp = pd.concat(shard_pred, ignore_index=True)
            sp.to_parquet(tmp_p, index=False)
            pd.DataFrame(shard_metrics).to_csv(tmp_m, index=False)
            tmp_p.replace(ppath)
            tmp_m.replace(mpath)
            built.append((label, h))
            print(
                f"  BUILT [{shard_no:02d}/{n_shards_total}] {label} h{int(h):02d}: "
                f"{len(sp):,} rows -> {project_relative(ppath)}",
                flush=True,
            )
            del shard_pred, shard_metrics, sp
            gc.collect()

    present = [(l, h) for (l, h) in expected if shard_pred_path(l, h).exists() and shard_metrics_path(l, h).exists()]
    missing = [(l, h) for (l, h) in expected if (l, h) not in present]
    print(
        f"\nShards complete: {len(present)}/{len(expected)} | built this run: {len(built)} | "
        f"skipped: {len(skipped)} | missing: {len(missing)}"
    )
    if missing:
        print(
            "Final combined outputs NOT written (incomplete shards):",
            [f"{l}_h{int(h):02d}" for l, h in missing][:12],
        )
        return None, None

    need_write = FORCE_OVERWRITE or bool(built) or (not PREDICTIONS_PATH.exists()) or (not METRICS_PATH.exists())
    if not need_write:
        print(
            "All shards present and final combined outputs already exist; not overwriting "
            "(FORCE_OVERWRITE=False and no new shards built this run)."
        )
        return None, None

    # Combine completed shards into final outputs once all expected shards exist.
    predictions = pd.concat(
        [pd.read_parquet(shard_pred_path(l, h)) for (l, h) in expected],
        ignore_index=True,
    )
    predictions.to_parquet(PREDICTIONS_PATH, index=False)
    per_origin = pd.concat(
        [pd.read_csv(shard_metrics_path(l, h)) for (l, h) in expected],
        ignore_index=True,
    )
    per_origin.to_csv(METRICS_PATH, index=False)

    def pooled(df, keys):
        return (
            df.groupby(keys, observed=True)
            .apply(
                lambda d: pd.Series(
                    {
                        "n": len(d),
                        "rmse": rmse(d["y_true"], d["y_pred"]),
                        "mae": mae(d["y_true"], d["y_pred"]),
                        "wape": wape(d["y_true"], d["y_pred"]),
                        "bias": bias(d["y_true"], d["y_pred"]),
                    }
                ),
                include_groups=False,
            )
            .reset_index()
        )

    by_hfs = (
        pooled(predictions, ["feature_set", "horizon"])
        .sort_values(["feature_set", "horizon"])
        .reset_index(drop=True)
    )
    by_hfs.to_csv(METRICS_BY_HFS_PATH, index=False)

    rmse_p = by_hfs.pivot(index="horizon", columns="feature_set", values="rmse").reset_index()
    if {"M1", "M2", "M3"}.issubset(set(rmse_p.columns)):
        rmse_p["m2_vs_m1_rmse_improvement"] = rmse_p["M1"] - rmse_p["M2"]
        rmse_p["m2_vs_m1_rmse_improvement_pct"] = 100 * (rmse_p["M1"] - rmse_p["M2"]) / rmse_p["M1"]
        rmse_p["m3_vs_m2_rmse_gap_positive_means_m3_better"] = rmse_p["M2"] - rmse_p["M3"]
    rmse_p.to_csv(GAIN_SUMMARY_PATH, index=False)

    pd.DataFrame([
        {
            "run_id": RUN_ID,
            "elapsed_minutes": (time.time() - t0) / 60.0,
            "n_predictions": len(predictions),
            "n_shards_built_this_run": len(built),
            "n_shards_total": len(expected),
        }
    ]).to_csv(RUNTIME_SUMMARY_PATH, index=False)
    print(f"\nFinal combine complete: {len(predictions):,} predictions from {len(expected)} shards.")
    print("Wrote:", project_relative(PREDICTIONS_PATH))
    print(by_hfs.round(3).to_string(index=False))
    return predictions, by_hfs


if EXECUTE_FINAL_RUN:
    predictions, by_hfs = run_final_evaluation()
else:
    print("EXECUTE_FINAL_RUN is False -> heavy rolling-origin training skipped.")
    print(
        f"Checkpointing by feature_set x horizon: shards under "
        f"{project_relative(PREDICTION_SHARDS_DIR)} and {project_relative(METRICS_SHARDS_DIR)}."
    )
    print("Completed shards are skipped on rerun (unless FORCE_OVERWRITE=True), so the run is crash-resumable.")
    print("Set EXECUTE_FINAL_RUN = True to launch the final run.")


## Summary

This cell prints the final daily-origin configuration, selected weather scenario, output folder, estimated runtime and execution flags.


In [ ]:
print("=" * 64)
print("09d FINAL DAILY-ORIGIN M1/M2/M3 - SUMMARY")
print("=" * 64)
print("Run ID:", RUN_ID)
print("Feature sets:", FEATURE_SETS, "(M4 excluded)")
print("Origin schedule:", ORIGIN_SCHEDULE_MODE, "| origins:", len(origin_schedule))
print(
    "M2 weather: conservative",
    M2_WEATHER_SCENARIO,
    f"(alpha h3={conservative_alpha(3):.4f}, h10={conservative_alpha(10):.4f})",
)
print("Conservative weather artifact:", project_relative(CONSERVATIVE_WEATHER_PATH))
print("Baseline operational weather left unchanged:", project_relative(BASELINE_OPERATIONAL_PATH))
print("Output folder:", project_relative(OUTPUT_ROLLING_RUN_DIR))
print(
    "Estimated fits / runtime:",
    f"{int(len(origin_schedule) * len(HORIZONS_TO_RUN) * len(FEATURE_SETS)):,}",
    f"/ ~{len(origin_schedule) * len(HORIZONS_TO_RUN) * len(FEATURE_SETS) * EST_MINUTES_PER_FIT / 60:.1f}h",
)
print("EXECUTE_FINAL_RUN:", EXECUTE_FINAL_RUN, "| FORCE_OVERWRITE:", FORCE_OVERWRITE)